In [1]:
import numpy as np
import plotly.graph_objects as go

rng = np.random.default_rng(7)

# ---- "Reality": data from a 2-component GMM ----
w = np.array([0.45, 0.55])
mu = np.array([-1.3,  1.9])
sd = np.array([0.55, 0.85])

N = 900
z = rng.choice(2, size=N, p=w)
x = rng.normal(mu[z], sd[z])

def normal_pdf(t, m, s):
    return (1.0/(s*np.sqrt(2*np.pi))) * np.exp(-0.5*((t-m)/s)**2)

xmin, xmax = float(x.min() - 1.5), float(x.max() + 1.5)
grid = np.linspace(xmin, xmax, 500)

pdf1 = normal_pdf(grid, mu[0], sd[0])
pdf2 = normal_pdf(grid, mu[1], sd[1])
mix  = w[0]*pdf1 + w[1]*pdf2

# ---- Wrong model: single Gaussian MLE ----
m_hat = x.mean()
s_hat = x.std(ddof=0)
single = normal_pdf(grid, m_hat, s_hat)

# ---- Fixed axis ranges (prevents zooming/autoscale on toggles) ----
hist_y, _ = np.histogram(x, bins=45, range=(xmin, xmax), density=True)
ymax = float(max(hist_y.max(), pdf1.max(), pdf2.max(), mix.max(), single.max()) * 1.15)

fig = go.Figure()

# Observations (histogram)
fig.add_trace(go.Histogram(
    x=x, nbinsx=45, histnorm="probability density",
    opacity=0.35, name="Observations (histogram)"
))

# Curves
i_single = len(fig.data)
fig.add_trace(go.Scatter(
    x=grid, y=single, mode="lines",
    line=dict(dash="dash"),
    name="Single Gaussian (wrong)",
    visible=True   # ON by default
))

i_c1 = len(fig.data)
fig.add_trace(go.Scatter(
    x=grid, y=pdf1, mode="lines",
    name="Component 1",
    visible=False
))

i_c2 = len(fig.data)
fig.add_trace(go.Scatter(
    x=grid, y=pdf2, mode="lines",
    name="Component 2",
    visible=False
))

i_mix = len(fig.data)
fig.add_trace(go.Scatter(
    x=grid, y=mix, mode="lines",
    name="Mixture (GMM)",
    visible=False
))

def toggle_button(label, trace_index):
    # args/args2 makes it a true toggle button
    return dict(
        label=label,
        method="restyle",
        args=[{"visible": [True]}, [trace_index]],
        args2=[{"visible": [False]}, [trace_index]],
    )

fig.update_layout(
    height=420,
    margin=dict(l=30, r=30, t=20, b=40),
    barmode="overlay",
    xaxis=dict(title="x", range=[xmin, xmax], fixedrange=True),
    yaxis=dict(title="density", range=[0, ymax], fixedrange=True),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    # prevents UI resets across re-renders
    uirevision="locked",
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            x=0.25, y=1.0,
            xanchor="left", yanchor="top",
            showactive=True,
            buttons=[
                toggle_button("Toggle single Gaussian", i_single),
                toggle_button("Toggle component 1", i_c1),
                toggle_button("Toggle component 2", i_c2),
                toggle_button("Toggle mixture", i_mix),
            ],
        )
    ],
)

fig